In [1]:
import json
import os

meta_file = "/qumulo/shared_data/aofei_summer/data/BiomedParse_meta.json"

In [2]:
# If meta_file is provided, load multiple datasets
with open(meta_file, 'r') as f:
    meta = json.load(f)
roots = meta['roots']
mask_roots = meta['mask_roots']
json_paths = meta['json_paths']
modality_labels = meta['modality_labels']
assert len(roots) == len(mask_roots) == len(json_paths) == len(modality_labels)

imgid_to_anns = {}
image_infos = {}
categories = None
num_classes = None
all_annotations = []

for r, m, j, modality in zip(roots, mask_roots, json_paths, modality_labels):
    # replace the "train" as "test" in the json file path   
    j = j.replace("train", "test")
    r = r.replace("train", "test")
    m = m.replace("train", "test")
    with open(j, 'r') as f:
        data = json.load(f)
    annotations = data['annotations']
    categories = data['categories']
    if categories is None:
        categories = categories
        num_classes = len(categories)
    # Build mapping from image_id to all masks and info
    for ann in annotations:
        img_id = ann['image_id']
        # Make img_id unique across datasets by prefixing with dataset index
        unique_img_id = f"{r}_{img_id}"
        ann['image_id'] = unique_img_id
        ann['file_name'] = os.path.join(r, ann['file_name'])
        ann['mask_file'] = os.path.join(m, ann['mask_file'])
        ann['modality_label'] = int(modality) if modality is not None else -1
        if unique_img_id not in imgid_to_anns:
            imgid_to_anns[unique_img_id] = []
        imgid_to_anns[unique_img_id].append(ann)
    all_annotations.extend(annotations)
    for ann in annotations:
        img_id = ann['image_id']
        if img_id not in image_infos:
            image_infos[img_id] = ann['file_name']

print(f"Total images: {len(imgid_to_anns)}, total annotations: {len(all_annotations)}")
# filter annotations to include more than 0 masks
filtered_annotations = [ann for ann in all_annotations if len(imgid_to_anns[ann['image_id']]) > 0]
print(f"Filtered annotations: {len(filtered_annotations)}")
annotations = filtered_annotations
image_ids = list(image_infos.keys())

Total images: 54858, total annotations: 116793
Filtered annotations: 116793


In [ ]:
image_morethan_one_mask_flag = {}

In [3]:
len(set([ann['image_id'] for ann in annotations]))  # number of modalities

54858

In [4]:
annotations[0]

{'mask_file': '/qumulo/shared_data/aofei_summer/data/BiomedParse/amos22/amos22/CT/test_mask/amos_0333_112_CT_abdomen_aorta.png',
 'area': 3188,
 'iscrowd': 0,
 'image_id': '/qumulo/shared_data/aofei_summer/data/BiomedParse/amos22/amos22/CT/test_0',
 'bbox': [426, 538, 75, 55],
 'category_id': 8,
 'id': 0,
 'slice_ratio': 0.3,
 'file_name': '/qumulo/shared_data/aofei_summer/data/BiomedParse/amos22/amos22/CT/test/amos_0333_112_CT_abdomen.png',
 'split': 'test',
 'sentences': [{'raw': 'aorta in abdominal computed tomography',
   'sent': 'aorta in abdominal computed tomography',
   'sent_id': 0},
  {'raw': 'aorta in abdominal CT',
   'sent': 'aorta in abdominal CT',
   'sent_id': 1}],
 'sent_ids': [0, 1],
 'ann_id': 0,
 'ref_id': 0,
 'modality_label': 0}

In [5]:
# get the stats of numbers of different datasets
dataset_stats = {}
dataset_image_stats = {}
for ann in all_annotations:
    dataset_name = ann['file_name'].split('/')[6]
    image_id = ann['image_id']
    if dataset_name == "amos22":
        dataset_name = "amos22" + "/" + ann['file_name'].split('/')[8]
    if dataset_name == "MSD" or dataset_name == "Radiography":
        dataset_name = ann['file_name'].split('/')[-3]


    if dataset_name not in dataset_stats:
        dataset_stats[dataset_name] = 0
        dataset_image_stats[dataset_name] = []
    dataset_stats[dataset_name] += 1
    dataset_image_stats[dataset_name].append(image_id)

# print("Dataset statistics:")
# for ds, count in dataset_stats.items():
#     print(f"  {ds}: {count}") 

print("Dataset image statistics:")
for img, count in dataset_image_stats.items():
    print(f"  {img}: {len(set(count))}")

sampled_nums = {
    "amos22/CT": (0, "CT"),
    "amos22/MRI": (0, "MRI"),
    "BreastUS": (128, "US"),
    "DRIVE": (5, "Fundus"),
    "CAMUS": (200, "US"),
    "CDD-CESM": (200, "X-ray"),
    "CXR_Masks_and_Labels": (22, "X-ray"),
    "FH-PS-AOP": (100, "US"),
    "G1020": (204, "Fundus"),
    "GlaS": (42, "Pathology"),
    "ISIC": (300, "Dermoscopy"),
    "kits23": (200, "CT"),
    "LGG": (200, "MRI"),
    "LIDC-IDRI": (300, "CT"),
    "LiverUS": (9, "US"),
    "MMs": (200, "MRI"),
    "Task01_BrainTumour": (300, "MRI"),
    "Task02_Heart": (100, "MRI"),
    "Task03_Liver": (200, "CT"),
    "Task04_Hippocampus": (200, "MRI"),
    "Task05_Prostate": (200, "MRI"),
    "Task06_Lung": (200, "CT"),
    "Task07_Pancreas": (100, "CT"),
    "Task08_HepaticVessel": (80, "CT"),
    "Task09_Spleen": (100, "CT"),
    "Task10_Colon": (100, "CT"),
    "NeoPolyp": (100, "Endoscopy"),
    "OCT-CME": (283, "OCT"),
    "PanNuke": (100, "Pathology"),
    "PolypGen": (100, "Endoscopy"),
    "COVID-19_CT": (50, "CT"),
    "COVID-QU-Ex": (100, "X-ray"),
    "QaTa-COV19": (100, "X-ray"),
    "COVID": (100, "X-ray"),
    "Lung_Opacity": (100, "X-ray"),
    "Normal": (50, "X-ray"),
    "Viral_Pneumonia": (100, "X-ray"),
    "REFUGE": (100, "Fundus"),
    "siim-acr-pneumothorax": (150, "X-ray"),
    "UWaterlooSkinCancer": (41, "Dermatoscopy")

}

Dataset image statistics:
  amos22/CT: 8349
  amos22/MRI: 490
  BreastUS: 128
  CAMUS: 4164
  CDD-CESM: 217
  CXR_Masks_and_Labels: 22
  DRIVE: 5
  FH-PS-AOP: 800
  G1020: 204
  GlaS: 42
  ISIC: 1000
  kits23: 5807
  LGG: 257
  LIDC-IDRI: 1733
  LiverUS: 9
  MMs: 630
  Task01_BrainTumour: 9831
  Task02_Heart: 248
  Task03_Liver: 4115
  Task04_Hippocampus: 1214
  Task05_Prostate: 186
  Task06_Lung: 242
  Task07_Pancreas: 1699
  Task08_HepaticVessel: 2357
  Task09_Spleen: 198
  Task10_Colon: 245
  NeoPolyp: 200
  OCT-CME: 283
  PanNuke: 1254
  PolypGen: 299
  COVID-19_CT: 385
  COVID-QU-Ex: 1166
  QaTa-COV19: 2113
  COVID: 724
  Lung_Opacity: 1203
  Normal: 2039
  Viral_Pneumonia: 269
  REFUGE: 400
  siim-acr-pneumothorax: 290
  UWaterlooSkinCancer: 41


In [6]:

dataset_sampled_num = sampled_nums
dataset_sampled_num

{'amos22/CT': (0, 'CT'),
 'amos22/MRI': (0, 'MRI'),
 'BreastUS': (128, 'US'),
 'DRIVE': (5, 'Fundus'),
 'CAMUS': (200, 'US'),
 'CDD-CESM': (200, 'X-ray'),
 'CXR_Masks_and_Labels': (22, 'X-ray'),
 'FH-PS-AOP': (100, 'US'),
 'G1020': (204, 'Fundus'),
 'GlaS': (42, 'Pathology'),
 'ISIC': (300, 'Dermoscopy'),
 'kits23': (200, 'CT'),
 'LGG': (200, 'MRI'),
 'LIDC-IDRI': (300, 'CT'),
 'LiverUS': (9, 'US'),
 'MMs': (200, 'MRI'),
 'Task01_BrainTumour': (300, 'MRI'),
 'Task02_Heart': (100, 'MRI'),
 'Task03_Liver': (200, 'CT'),
 'Task04_Hippocampus': (200, 'MRI'),
 'Task05_Prostate': (200, 'MRI'),
 'Task06_Lung': (200, 'CT'),
 'Task07_Pancreas': (100, 'CT'),
 'Task08_HepaticVessel': (80, 'CT'),
 'Task09_Spleen': (100, 'CT'),
 'Task10_Colon': (100, 'CT'),
 'NeoPolyp': (100, 'Endoscopy'),
 'OCT-CME': (283, 'OCT'),
 'PanNuke': (100, 'Pathology'),
 'PolypGen': (100, 'Endoscopy'),
 'COVID-19_CT': (50, 'CT'),
 'COVID-QU-Ex': (100, 'X-ray'),
 'QaTa-COV19': (100, 'X-ray'),
 'COVID': (100, 'X-ray'),
 'Lun

In [7]:
import random
all_Sampled_images = []
for ds, count in dataset_stats.items():
    all_image_names = list(set(dataset_image_stats[ds]))
    
    if len(all_image_names) > dataset_sampled_num[ds][0]:
        sampled_images = random.sample(all_image_names, dataset_sampled_num[ds][0]//2)
    else:
        sampled_images = all_image_names
    sampled_images_With_modality = [(img, dataset_sampled_num[ds][1]) for img in sampled_images]
    all_Sampled_images.extend(sampled_images_With_modality)

In [11]:
len(all_Sampled_images)

3035

In [11]:
all_Sampled_images[:3]  # list of (image_id, modality) pairs

[('/qumulo/shared_data/aofei_summer/data/BiomedParse/BreastUS/BreastUS/test_113',
  'US'),
 ('/qumulo/shared_data/aofei_summer/data/BiomedParse/BreastUS/BreastUS/test_25',
  'US'),
 ('/qumulo/shared_data/aofei_summer/data/BiomedParse/BreastUS/BreastUS/test_82',
  'US')]

In [12]:
# len(all_Sampled_images)
# all_Sampled_images[0]
step1_sampled_data = dict()
for k in all_Sampled_images:
    k, modality = k
    step1_sampled_data[k] = dict()
    step1_sampled_data[k]['image_file'] = image_infos[k]
    step1_sampled_data[k]['mask_annotations'] = imgid_to_anns[k]
    if "BreastUS" in k:
        step1_sampled_data[k]['mask_annotations'] = [imgid_to_anns[k][0]]
    step1_sampled_data[k]['modality'] = modality

In [13]:
list(step1_sampled_data.items())[0]

('/qumulo/shared_data/aofei_summer/data/BiomedParse/BreastUS/BreastUS/test_27',
 {'image_file': '/qumulo/shared_data/aofei_summer/data/BiomedParse/BreastUS/BreastUS/test/benign (46)_ultrasound_breast.png',
  'mask_annotations': [{'mask_file': '/qumulo/shared_data/aofei_summer/data/BiomedParse/BreastUS/BreastUS/test_mask/benign (46)_ultrasound_breast_benign+tumor.png',
    'area': 8103,
    'iscrowd': 0,
    'image_id': '/qumulo/shared_data/aofei_summer/data/BiomedParse/BreastUS/BreastUS/test_27',
    'bbox': [305, 414, 87, 109],
    'category_id': 10,
    'id': 54,
    'slice_ratio': 1.0,
    'file_name': '/qumulo/shared_data/aofei_summer/data/BiomedParse/BreastUS/BreastUS/test/benign (46)_ultrasound_breast.png',
    'split': 'test',
    'sentences': [{'raw': 'benign tumor',
      'sent': 'benign tumor',
      'sent_id': 114}],
    'sent_ids': [114],
    'ann_id': 54,
    'ref_id': 54,
    'modality_label': 2}],
  'modality': 'US'})

In [14]:
len(step1_sampled_data)

3035

In [15]:
# save the results
out_sample_file = "/qumulo/shared_data/aofei_summer/RegTok/data/BiomedParse_SegVQA_Diagnosis_30k.json"
with open(out_sample_file, "w") as f:
    json.dump(step1_sampled_data, f, indent=2)

In [ ]:
# quantizer_infos = []
# all_pred_classes = []
# all_gt_classes = []
# all_gt_masks = []
# all_original_images = []

# keys = sampled_ids
# batch_size = 2
# for k in tqdm(range(0, len(keys), batch_size)):
#     batch_keys = keys[k : k + batch_size]
#     images = []
#     masks_batch = []
#     class_labels_batch = []
#     text_embeddings_batch = []
#     modality_labels = []

#     # load each sample in the batch
#     for img_id in batch_keys:
#         image, masks, class_labels, text_embeddings, modality_label = get_image_data(img_id)
#         images.append(image)                       # (1, C, H, W)
#         masks_batch.append(masks)                  # (num_masks, H, W) or (0,...)
#         class_labels_batch.append(class_labels)    # (num_masks,) or (0,)
#         text_embeddings_batch.append(text_embeddings)
#         modality_labels.append(modality_label)

#     imgs = torch.cat([item for item in images], dim=0)
#     masks = [item.squeeze(1) for item in masks_batch]
#     class_labels = [item for item in class_labels_batch]

#     imgs = imgs.to(device)
#     masks = [i.to(device) for i in masks]


#     with torch.no_grad():
#         outputs = vq_model(
#             imgs, do_quantize=True, mask_labels=masks, class_labels=[c.to(device) for c in class_labels], loss_type="dice_bce"
#         )
#         dec, diff, dice_loss, bce_loss, cls_loss, seg_logits, class_logits, hierarchical_codes, hierarchical_masks, hierarchical_gt_masks, hierarchical_losses, quantization_losses, \
#             total_quantization_loss, dice_loss_normal, dice_loss_quant, cls_loss_normal, cls_loss_quant, distill_loss, quantizer_info, semantic_loss = outputs
#         quantizer_infos.append(quantizer_info)
#         # print(quantizer_info)
#         # Optionally collect class predictions for analysis
#         if class_logits is not None:
#             all_pred_classes.append([logit.argmax().item() for logit in class_logits[0]])
#         #     all_gt_classes.append([c.item() for c in class_labels[0]])
#         # all_gt_masks.append([m.cpu().numpy() for m in masks[0]])
#         for b in range(len(class_labels)):
#             all_gt_classes.append([c.item() for c in class_labels[b]])
#             all_gt_masks.append([m.cpu().numpy() for m in masks[b]])
#             all_original_images.append(imgs[b].cpu().numpy())
#     codebook_indices = quantizer_info['indices']  # List of (batch_size, num_queries) per stage
#     hungarian_indices = quantizer_info['hungarian_indices']  # List of (batch_size, num_queries_matches) per stage
#     modality_labels_pred = quantizer_info['modality_label_pred']  # (batch_size,)
    
#     for m in range(batch_size):
#         image_id_for_write_back = batch_keys[m]
#         target_for_write_back = step1_sampled_data[image_id_for_write_back]

#         codebook_indices_per_image = codebook_indices[m].cpu().numpy()
#         hungarian_indices_per_image = hungarian_indices[m]
#         # num_matched = len(hungarian_indices_per_image) // 2
#         num_matched = hungarian_indices_per_image[0].shape[0]
#         modality_label_for_write_back = modality_labels_pred[m]
#         for p in range(num_matched):
#             query_idx = hungarian_indices_per_image[0].cpu().numpy()[p]
#             codebook_idx = codebook_indices_per_image[query_idx]
#             target_for_write_back['mask_annotations'][p]['quantizer_code'] = f"M{modality_label_for_write_back}_{codebook_idx}"


In [16]:
batch_keys, hungarian_indices,codebook_indices

(['/qumulo/shared_data/aofei_summer/data/BiomedParse/MSD/MSD/Task03_Liver/train_48',
  '/qumulo/shared_data/aofei_summer/data/BiomedParse/REFUGE/REFUGE/train_661'],
 [(tensor([ 3, 13]), tensor([0, 1])), (tensor([12, 13]), tensor([0, 1]))],
 tensor([[28, 16, 25, 16, 25, 25, 25, 28, 15, 16, 16, 16, 25, 25, 15, 25, 25, 25,
          25, 25],
         [19, 21, 15,  0,  6,  0, 11, 29, 20, 24, 21,  3,  0,  6, 20,  0, 21,  0,
           0,  6]]))

In [17]:
step1_sampled_data['/qumulo/shared_data/aofei_summer/data/BiomedParse/MSD/MSD/Task03_Liver/train_48']

{'image_file': '/qumulo/shared_data/aofei_summer/data/BiomedParse/MSD/MSD/Task03_Liver/train/liver_1_72_CT_liver.png',
 'mask_annotations': [{'mask_file': '/qumulo/shared_data/aofei_summer/data/BiomedParse/MSD/MSD/Task03_Liver/train_mask/liver_1_72_CT_liver_liver.png',
   'area': 53056,
   'iscrowd': 0,
   'image_id': '/qumulo/shared_data/aofei_summer/data/BiomedParse/MSD/MSD/Task03_Liver/train_48',
   'bbox': [422, 508, 399, 231],
   'category_id': 1,
   'id': 72,
   'slice_ratio': 0.3,
   'file_name': '/qumulo/shared_data/aofei_summer/data/BiomedParse/MSD/MSD/Task03_Liver/train/liver_1_72_CT_liver.png',
   'split': 'train',
   'sentences': [{'raw': 'liver in liver computed tomography',
     'sent': 'liver in liver computed tomography',
     'sent_id': 165},
    {'raw': 'liver in liver CT', 'sent': 'liver in liver CT', 'sent_id': 166},
    {'raw': 'hepatic organ in liver CT',
     'sent': 'hepatic organ in liver CT',
     'sent_id': 167}],
   'sent_ids': [165, 166, 167],
   'ann_id': 

In [40]:
target_for_write_back

{'image_file': '/qumulo/shared_data/aofei_summer/data/BiomedParse/amos22/amos22/CT/train/amos_0406_81_CT_liver.png',
 'mask_annotations': [{'mask_file': '/qumulo/shared_data/aofei_summer/data/BiomedParse/amos22/amos22/CT/train_mask/amos_0406_81_CT_liver_liver.png',
   'area': 14117,
   'iscrowd': 0,
   'image_id': '/qumulo/shared_data/aofei_summer/data/BiomedParse/amos22/amos22/CT/train_10389',
   'bbox': [323, 557, 207, 105],
   'category_id': 1,
   'id': 35816,
   'slice_ratio': 0.1,
   'file_name': '/qumulo/shared_data/aofei_summer/data/BiomedParse/amos22/amos22/CT/train/amos_0406_81_CT_liver.png',
   'split': 'train',
   'sentences': [{'raw': 'liver in liver CT',
     'sent': 'liver in liver CT',
     'sent_id': 70226}],
   'sent_ids': [70226],
   'ann_id': 35816,
   'ref_id': 35816,
   'modality_label': 0,
   'quantizer_code': 'M0_16'}]}